# AAMS Week 3 – Analysis of Discrete-Time Dynamical Models

Exercise notebook covering all exercises from the Week 3 lecture slides:
- **Part I**: Phase Space & Trajectories (2D & 3D), Finding Equilibrium Points (Ex. 5.1 & 5.2)
- **Part II**: Cobweb Plots for 1D Iterative Maps (Ex. 5.6 & 5.7, Exponential Growth & Decay)
- **Part III**: Variable Rescaling & Dimensionless Maps (Ex. 5.9 & the Logistic Map)
- **Part IV**: Asymptotic Behaviour of Linear Systems (Linear System Explorer: $L_1, L_2, L_3$ & Ex. 5.12)
- **Part V**: Linear Stability Analysis of Nonlinear Systems (Ex. 5.15, 5.16 & 5.17)

**How it works:** Each section has a short theory part at the top, then a function with `# TODO: implement`,
then a test cell. The tests fail until your implementation is correct.
Hints are collapsed and graded – open only as many as you need.
The plotting helpers are finished so you can focus on the models.

Convention as in the lecture: $x_t = F(x_{t-1})$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

plt.rcParams["figure.dpi"] = 110


def plot_timeseries(ts, labels=None, title="", logy=False, ax=None):
    ts = np.asarray(ts, dtype=float)
    if ts.ndim == 1:
        ts = ts[:, None]
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 3.5))
    for i in range(ts.shape[1]):
        ax.plot(ts[:, i], marker="o", ms=3, label=labels[i] if labels else None)
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("t")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    if labels:
        ax.legend()
    return ax


def plot_phase(ts_data, labels=("x", "y"), title="Phase space", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ts_list = ts_data if isinstance(ts_data, list) else [ts_data]
    for ts in ts_list:
        ts = np.asarray(ts, dtype=float)
        p = ax.plot(ts[:, 0], ts[:, 1], "-o", ms=3, lw=1, alpha=0.85)
        ax.plot(ts[0, 0], ts[0, 1], "o", color=p[0].get_color(), ms=6)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)
    ax.grid(alpha=0.3)
    return ax


def plot_phase_3d(ts_list, labels=("x", "y", "z"), title="3D Phase Space", ax=None):
    if ax is None:
        fig = plt.figure(figsize=(7, 6))
        ax = fig.add_subplot(111, projection="3d")
    if not isinstance(ts_list, list):
        ts_list = [ts_list]
    for ts in ts_list:
        ts = np.asarray(ts, dtype=float)
        line = ax.plot(ts[:, 0], ts[:, 1], ts[:, 2], "-o", ms=2, lw=1, alpha=0.8)[0]
        ax.scatter([ts[0, 0]], [ts[0, 1]], [ts[0, 2]], color=line.get_color(), s=25)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_zlabel(labels[2])
    ax.set_title(title)
    return ax


def plot_cobweb(f, path, xlim, title="", ax=None):
    path = np.asarray(path, dtype=float)
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    xs = np.linspace(*xlim, 400)
    ax.plot(xs, f(xs), "k", label=r"$x_t = f(x_{t-1})$")
    ax.plot(xlim, xlim, "k--", lw=0.8, label=r"$x_t = x_{t-1}$")
    ax.plot([path[0, 0], path[0, 0]], [xlim[0], path[0, 1]], "r", lw=1)
    ax.plot(path[:, 0], path[:, 1], "r", lw=1, label="Cobweb")
    ax.set_xlim(xlim)
    ax.set_ylim(xlim)
    ax.set_xlabel(r"$x_{t-1}$")
    ax.set_ylabel(r"$x_t$")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
    return ax

## 0 · The Simulator

A first-order difference equation is simulated by applying the update rule $T$ times, starting from the
initial state $x_0$. The result is a time series $\{x_0, x_1, \dots, x_T\}$.
You will need this simulator in almost every section below – build it once, cleanly.

**Contract:** `step` takes a state (scalar or vector) and returns the next one.
The return value is an `np.ndarray` of shape `(T+1,)` for scalars or `(T+1, dim)` for vectors, including the initial state.

In [ ]:
def simulate(step, state0, T):
    '''Returns the time series of all states from t=0 to t=T.'''
    state0 = np.asarray(state0, dtype=float)
    ts = np.empty((T + 1,) + state0.shape, dtype=float)
    ts[0] = state0
    for t in range(1, T + 1):
        ts[t] = step(ts[t - 1])
    return ts

In [ ]:
# Tests
assert np.allclose(simulate(lambda x: 2 * x, 1, 3), [1, 2, 4, 8])
assert simulate(lambda x: 2 * x, 1, 3).shape == (4,)
out = simulate(lambda s: s + 1, np.array([0, 10]), 2)
assert out.shape == (3, 2) and np.allclose(out, [[0, 10], [1, 11], [2, 12]])
print("✓ simulate")

## 1 · 2D Phase Space: Trajectories & Sensitivity to Initial Conditions (Slides 8–11)

Phase space represents the complete state of a dynamical system geometrically.
Consider the 2D linear discrete dynamical system:
$$x_t = 0.5\,x_{t-1} + y_{t-1}$$
$$y_t = -0.5\,x_{t-1} + y_{t-1}$$

In matrix form: $\mathbf{s}_t = A\,\mathbf{s}_{t-1}$ with $\mathbf{s} = \begin{pmatrix} x \\ y \end{pmatrix}$.

**Tasks:**
1. Implement `step_linear_2d(state)` updating both state variables simultaneously.
2. Provide the coefficient matrix `A_linear_2d`.
3. Study the phase portraits when varying $x_0$ and $(x_0, y_0)$.

In [ ]:
def step_linear_2d(state):
    '''One time step of the 2D system; state = (x, y).'''
    # TODO: implement
    raise NotImplementedError


A_linear_2d = None  # TODO: 2x2 numpy array representing the system matrix

In [ ]:
# Tests
first_step = step_linear_2d(np.array([1.0, 1.0]))
assert np.allclose(first_step, [1.5, 0.5]), f"Expected [1.5, 0.5], got {first_step}"
assert not np.allclose(first_step, [1.5, 0.25]), "Watch out: y must be computed with the OLD x!"
assert A_linear_2d is not None, "A_linear_2d must be defined"
assert np.allclose(A_linear_2d @ np.array([1.0, 1.0]), [1.5, 0.5])
print("✓ step_linear_2d & A_linear_2d")

# Visualization 1: Slide 9 (varying x_0 with y_0 = 1.0)
x0_vals = np.arange(0.0, 2.25, 0.25)
trajs_x0 = [simulate(step_linear_2d, [x0, 1.0], 20) for x0 in x0_vals]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for x0, ts in zip(x0_vals, trajs_x0):
    axes[0].plot(ts[:, 0], marker="o", ms=2.5, label=f"IC: ({x0}, 1.0)")
axes[0].set_title("Time Series $x_t$ (Slide 9)")
axes[0].set_xlabel("t"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=7, ncol=2)
plot_phase(trajs_x0, title="Phase Space: $x_0$ Variation (Slide 9)", ax=axes[1])
plt.tight_layout()

# Visualization 2: Slide 10 & 11 (varying both x_0 and y_0)
ics = [(1.0, 1.0), (-1.0, 1.0), (-1.0, -1.0), (1.0, -1.0), (0.5, 0.0), (0.0, -1.5)]
trajs_ics = [simulate(step_linear_2d, ic, 20) for ic in ics]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ic, ts in zip(ics, trajs_ics):
    axes[0].plot(ts[:, 0], marker="o", ms=2.5, label=f"IC: {ic}")
axes[0].set_title("Time Series $x_t$ (Slide 10)")
axes[0].set_xlabel("t"); axes[0].grid(alpha=0.3); axes[0].legend(fontsize=7)
plot_phase(trajs_ics, title="Phase Space: $(x_0, y_0)$ Variation (Slide 10/11)", ax=axes[1])
plt.tight_layout();

<details><summary>Hint 1</summary>

In difference equations, all variables update simultaneously. Unpack `x, y = state` first, then calculate `x_next` and `y_next` using only the old variables.

</details>

<details><summary>Hint 2</summary>

For the matrix $A$, row 0 holds the coefficients for $x_{t-1}$ and $y_{t-1}$ in the $x_t$ equation: `[0.5, 1.0]`. Row 1 holds those for $y_t$: `[-0.5, 1.0]`.

</details>

<details><summary>Interpretation</summary>

The matrix $A$ has $\det A = (0.5)(1.0) - (1.0)(-0.5) = 1.0$ and $\text{Tr}(A) = 1.5$. The eigenvalues are complex conjugates:
$$\lambda_{1,2} = 0.75 \pm 0.6614i, \quad |\lambda_{1,2}| = 1.0$$
Because $|\lambda| = 1.0$, the orbits are closed ellipses (neutral center). Each initial state stays on its own invariant orbit forever without decaying or exploding.

</details>

## 2 · 3D Phase Space (Slide 12)

Consider the three-dimensional discrete-time system:
$$x_t = 0.5\,x_{t-1} + y_{t-1}$$
$$y_t = -0.5\,x_{t-1} + y_{t-1}$$
$$z_t = -x_{t-1} - y_{t-1} + z_{t-1}$$

**Tasks:**
1. Implement `step_3d(state)` and the matrix `A_3d`.
2. Simulate and visualize the trajectories in 3D phase space for initial conditions in $[-2, 2]^3$.

In [ ]:
def step_3d(state):
    '''One step of the 3D difference equation; state = (x, y, z).'''
    # TODO: implement
    raise NotImplementedError


A_3d = None  # TODO: 3x3 numpy array

In [ ]:
# Tests
s_next = step_3d([1.0, 1.0, 1.0])
assert np.allclose(s_next, [1.5, 0.5, -1.0]), f"Expected [1.5, 0.5, -1.0], got {s_next}"
assert A_3d is not None and A_3d.shape == (3, 3)
assert np.allclose(A_3d @ np.array([1.0, 1.0, 1.0]), [1.5, 0.5, -1.0])
assert np.allclose(step_3d([0.0, 0.0, 2.0]), [0.0, 0.0, 2.0]), "z-axis should be an invariant line"
print("✓ step_3d & A_3d")

# 3D Phase Space Visualization (Slide 12)
ics_3d = [
    [1.0, 1.0, 1.0],
    [-1.0, 1.0, 0.5],
    [-1.0, -1.0, -1.0],
    [1.0, -1.0, -0.5],
    [0.5, 0.0, 2.0],
    [0.0, -1.5, -2.0],
]
trajs_3d = [simulate(step_3d, ic, 25) for ic in ics_3d]
plot_phase_3d(trajs_3d, title="3D Phase Space (Slide 12)");

<details><summary>Hint</summary>

The third equation is $z_t = -1.0 x_{t-1} - 1.0 y_{t-1} + 1.0 z_{t-1}$. In matrix form, row 2 is `[-1.0, -1.0, 1.0]`.

</details>

<details><summary>Interpretation</summary>

Notice the block structure: $x$ and $y$ are completely independent of $z$. In the $xy$-plane, the system orbits an ellipse. Meanwhile, $z$ accumulates a drift driven by $-(x + y)$. Along the $z$-axis ($x=0, y=0$), $z_t = z_{t-1}$, so every point on the $z$-axis is an equilibrium point.

</details>

## 3 · Finding Equilibrium Points (Exercises 5.1 & 5.2 – Slides 6–7)

An equilibrium point $x_{eq}$ satisfies the fixed-point condition:
$$x_{eq} = F(x_{eq})$$

**Exercise 5.1:**
$$x_t = 2\,x_{t-1} - x_{t-1}^2$$

**Exercise 5.2:**
$$x_t = x_{t-1}\,y_{t-1}$$
$$y_t = y_{t-1}\,(x_{t-1} - 1)$$

**Tasks:**
1. Implement `equilibria_5_1()` returning the fixed point(s) as a sorted list/array.
2. Implement `equilibria_5_2()` returning the equilibrium pairs $(x_{eq}, y_{eq})$.
3. Implement the general validator `is_equilibrium(step_func, state)`.

In [ ]:
def equilibria_5_1():
    '''Returns the equilibrium point(s) of x_t = 2*x_{t-1} - x_{t-1}^2 as a sorted list.'''
    # TODO: implement
    raise NotImplementedError


def equilibria_5_2():
    '''Returns the equilibrium point(s) of the 2D model as a list of (x, y) tuples.'''
    # TODO: implement
    raise NotImplementedError


def is_equilibrium(step_func, state, atol=1e-7):
    '''Checks whether step_func(state) == state within atol tolerance.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
eqs_51 = equilibria_5_1()
assert np.allclose(sorted(eqs_51), [0.0, 1.0]), f"Expected [0, 1], got {eqs_51}"
for eq in eqs_51:
    assert is_equilibrium(lambda x: 2 * x - x**2, eq)

step_52 = lambda s: np.array([s[0] * s[1], s[1] * (s[0] - 1)])
eqs_52 = sorted(equilibria_5_2())
assert len(eqs_52) == 2, f"Expected 2 equilibria, got {len(eqs_52)}"
assert np.allclose(eqs_52[0], (0.0, 0.0))
assert np.allclose(eqs_52[1], (2.0, 1.0))
for eq in eqs_52:
    assert is_equilibrium(step_52, eq)
print("✓ equilibria_5_1, equilibria_5_2 & is_equilibrium")

<details><summary>Hint: Ex. 5.1</summary>

Set $x_{eq} = 2x_{eq} - x_{eq}^2 \implies x_{eq}^2 - x_{eq} = 0 \implies x_{eq}(x_{eq} - 1) = 0$.

</details>

<details><summary>Hint: Ex. 5.2</summary>

From $x_{eq} = x_{eq} y_{eq}$, either $x_{eq} = 0$ or $y_{eq} = 1$.
- If $x_{eq} = 0$, the second equation $y_{eq} = y_{eq}(0 - 1) = -y_{eq}$ gives $2y_{eq} = 0 \implies y_{eq} = 0$.
- If $y_{eq} = 1$, the second equation $1 = 1(x_{eq} - 1)$ gives $x_{eq} = 2$.

</details>

## 4 · Cobweb Plots for 1D Iterative Maps (Exercises 5.6 & 5.7 – Slides 16–21)

A cobweb plot visualizes the path of a 1D difference equation by alternating:
1. **Vertically** from $(x_t, x_t)$ to the curve $(x_t, x_{t+1}) = (x_t, f(x_t))$.
2. **Horizontally** from $(x_t, x_{t+1})$ to the diagonal $(x_{t+1}, x_{t+1})$.

**Logistic growth model:**
$$N_t = N_{t-1} + r\,N_{t-1}\left(1 - \frac{N_{t-1}}{K}\right)$$

**Tasks:**
1. Implement `cobweb_path(f, x0, n)` returning the $2n + 1$ coordinate pairs:
   $$(x_0, x_0), (x_0, x_1), (x_1, x_1), (x_1, x_2), \dots, (x_n, x_n)$$
2. Implement `logistic_step(N, r, K)`.
3. Plot and compare Exponential Growth ($a=1.1, x_0=1$), Exponential Decay ($a=0.8, x_0=20$), Ex. 5.6 ($r=1$), and Ex. 5.7 ($r=2.5$).

In [ ]:
def cobweb_path(f, x0, n):
    '''Returns array of shape (2n+1, 2) with the alternating points on the cobweb path.'''
    # TODO: implement
    raise NotImplementedError


def logistic_step(N, r=1.0, K=1.0):
    '''One step of the discrete logistic growth model.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert np.allclose(cobweb_path(lambda x: 2 * x, 1, 2), [[1, 1], [1, 2], [2, 2], [2, 4], [4, 4]])
assert cobweb_path(np.cos, 0.3, 10).shape == (21, 2)
assert np.isclose(logistic_step(0.1, 1.0, 1.0), 0.19)
assert np.isclose(logistic_step(1.0, 1.0, 1.0), 1.0)
print("✓ cobweb_path & logistic_step")

# Four Cobweb Plots matching slides 20 & 21
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))

# 1. Exponential growth (a=1.1, x0=1)
f_grow = lambda x: 1.1 * x
plot_cobweb(f_grow, cobweb_path(f_grow, 1.0, 30), (0, 20), "Exp. Growth (a=1.1)", ax=axes[0])

# 2. Exponential decay (a=0.8, x0=20)
f_decay = lambda x: 0.8 * x
plot_cobweb(f_decay, cobweb_path(f_decay, 20.0, 20), (0, 20), "Exp. Decay (a=0.8)", ax=axes[1])

# 3. Ex. 5.6: Logistic growth (r=1.0, K=1.0, N0=0.1)
f_r1 = lambda x: logistic_step(x, r=1.0, K=1.0)
plot_cobweb(f_r1, cobweb_path(f_r1, 0.1, 15), (0, 1.05), "Ex. 5.6: Logistic (r=1)", ax=axes[2])

# 4. Ex. 5.7: Logistic growth (r=2.5, K=1.0, N0=0.1)
f_r25 = lambda x: logistic_step(x, r=2.5, K=1.0)
plot_cobweb(f_r25, cobweb_path(f_r25, 0.1, 50), (0, 1.4), "Ex. 5.7: Logistic (r=2.5)", ax=axes[3])
plt.tight_layout();

<details><summary>Hint</summary>

Initialize an array `pts = np.empty((2*n + 1, 2))`. Set `pts[0] = [x0, x0]`. Then loop `for i in range(n)`: calculate $x_{next} = f(x_{curr})$, record the vertical step `[x_{curr}, x_{next}]`, then the horizontal step `[x_{next}, x_{next}]`, and update $x_{curr} = x_{next}$.

</details>

<details><summary>Interpretation</summary>

- At $r = 1$, the slope at $N = 1$ is $f'(1) = 1 - r = 0 < 1$. The cobweb stairs in smoothly and stably.
- At $r = 2.5$, the slope at $N = 1$ is $1 - 2.5 = -1.5$. Since $|-1.5| > 1$, the fixed point is unstable. The cobweb spirals outward and locks into a stable periodic 2-cycle/4-cycle orbit.

</details>

## 5 · Variable Rescaling & Dimensionless Form (Ex. 5.9 – Slides 23–28)

Variable rescaling substitutes $x \to \alpha x'$ to eliminate parameters without altering the underlying dynamics.

**Exercise 5.9:** Simplify
$$x_t = \frac{a}{x_{t-1} + b}$$
By setting $x = \alpha x'$ with $\alpha = a/b$ and $c = a/b^2$, it reduces to:
$$x_t' = \frac{1}{c\,x_{t-1}' + 1}$$

**The Logistic Growth Model:**
$$x_t = x_{t-1} + r\,x_{t-1}\left(1 - \frac{x_{t-1}}{K}\right) \quad \xrightarrow{x = \alpha x'} \quad x_t' = r'\,x_{t-1}'(1 - x_{t-1}')$$
with $\alpha = \frac{K(1+r)}{r}$ and $r' = 1 + r$.

**Tasks:**
1. Implement `rescale_params_5_9(a, b)` returning $(\alpha, c)$.
2. Implement `step_5_9_original(x, a, b)` and `step_5_9_rescaled(x_prime, c)`.
3. Implement `rescale_params_logistic(r, K)` returning $(\alpha, r')$.

In [ ]:
def rescale_params_5_9(a, b):
    '''Returns (alpha, c) for the rescaled form x' = 1 / (c*x' + 1).'''
    # TODO: implement
    raise NotImplementedError


def step_5_9_original(x, a, b):
    '''One step of x_t = a / (x_{t-1} + b).'''
    # TODO: implement
    raise NotImplementedError


def step_5_9_rescaled(x_prime, c):
    '''One step of x'_t = 1 / (c * x'_{t-1} + 1).'''
    # TODO: implement
    raise NotImplementedError


def rescale_params_logistic(r, K):
    '''Returns (alpha, r_prime) mapping logistic growth to the logistic map.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests: Ex. 5.9
a, b = 4.0, 2.0
alpha, c = rescale_params_5_9(a, b)
assert np.isclose(alpha, 2.0) and np.isclose(c, 1.0)

# Check that alpha * x'_t == x_t along the entire trajectory
x0 = 3.0
ts_orig = simulate(lambda x: step_5_9_original(x, a, b), x0, 20)
ts_resc = simulate(lambda xp: step_5_9_rescaled(xp, c), x0 / alpha, 20)
assert np.allclose(alpha * ts_resc, ts_orig), "Rescaled trajectory must reconstruct original!"

# Tests: Logistic map rescaling
r, K = 1.5, 500.0
alpha_log, r_prime = rescale_params_logistic(r, K)
assert np.isclose(alpha_log, 500.0 * 2.5 / 1.5)
assert np.isclose(r_prime, 2.5)

logistic_map = lambda xp: r_prime * xp * (1.0 - xp)
ts_log_orig = simulate(lambda x: logistic_step(x, r, K), 50.0, 25)
ts_log_map = simulate(logistic_map, 50.0 / alpha_log, 25)
assert np.allclose(alpha_log * ts_log_map, ts_log_orig), "Logistic map trajectory must match scaled original!"
print("✓ variable rescaling (Ex. 5.9 & Logistic Map)")

<details><summary>Hint: Ex. 5.9 Derivation</summary>

$$\alpha x_t' = \frac{a}{\alpha x_{t-1}' + b} = \frac{1}{\frac{\alpha^2}{a}x_{t-1}' + \frac{\alpha b}{a}}$$
Setting $\frac{\alpha b}{a} = 1$ gives $\alpha = \frac{a}{b}$, which makes $c = \frac{\alpha^2}{a} = \frac{a}{b^2}$.

</details>

## 6 · Linear System Explorer: Eigenvalues & Stability (Slides 30–35)

For linear difference equations $\mathbf{x}_t = A\mathbf{x}_{t-1}$, the general solution is:
$$\mathbf{x}_t = \sum_{i=1}^n b_i \lambda_i^t \mathbf{v}_i$$

The **dominant eigenvalue** $\lambda_d$ (the one with largest absolute value $|\lambda_d|$) dictates the asymptotic behavior:
- $|\lambda_d| > 1$: **Unstable** (exponential divergence to $\infty$).
- $|\lambda_d| < 1$: **Stable** (exponential convergence to the origin $\mathbf{0}$).
- $|\lambda_d| = 1$: **Neutral / Marginally stable** (constant amplitude or sustained oscillations).

**Slide 34 Exercise:** Study the three matrices:
$$L_1 = \begin{pmatrix} 0 & 1.2 \\ 0.5 & 0.5 \end{pmatrix}, \quad L_2 = \begin{pmatrix} 0 & 1.0 \\ 0.5 & 0.5 \end{pmatrix}, \quad L_3 = \begin{pmatrix} 0 & 0.9 \\ 0.5 & 0.5 \end{pmatrix}$$

**Tasks:**
1. Define `L1`, `L2`, and `L3`.
2. Implement `dominant_eigenvalue(A)`.
3. Implement `classify_linear_system(A)` returning `"stable"`, `"neutral"`, or `"unstable"`.

In [ ]:
L1 = None  # TODO: matrix L1
L2 = None  # TODO: matrix L2
L3 = None  # TODO: matrix L3


def dominant_eigenvalue(A):
    '''Returns the eigenvalue of A with the largest absolute value.'''
    # TODO: implement
    raise NotImplementedError


def classify_linear_system(A, atol=1e-5):
    '''Returns "stable", "neutral", or "unstable" based on |lambda_d|.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert L1 is not None and L2 is not None and L3 is not None
d1 = abs(dominant_eigenvalue(L1))
d2 = abs(dominant_eigenvalue(L2))
d3 = abs(dominant_eigenvalue(L3))

assert d1 > 1.0 and classify_linear_system(L1) == "unstable"
assert np.isclose(d2, 1.0) and classify_linear_system(L2) == "neutral"
assert d3 < 1.0 and classify_linear_system(L3) == "stable"
print("✓ Linear System Explorer (L1, L2, L3)")

# Simulation & Visual Comparison (Slide 34)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
systems = [("L1 (Unstable)", L1), ("L2 (Neutral)", L2), ("L3 (Stable)", L3)]
for j, (name, A) in enumerate(systems):
    lam_d = dominant_eigenvalue(A)
    ts = simulate(lambda s: A @ s, [1.0, 1.0], 30)
    plot_timeseries(ts, labels=["x", "y"], title=f"{name}: $|\lambda_d|={abs(lam_d):.3f}$", ax=axes[0, j])
    plot_phase(ts, title=f"Phase: {name}", ax=axes[1, j])
plt.tight_layout();

<details><summary>Hint</summary>

Use `np.linalg.eigvals(A)` to retrieve all eigenvalues. To find the dominant one, locate `eigs[np.argmax(np.abs(eigs))]`.

</details>

<details><summary>Eigenvalue Solution</summary>

- $L_1$: $\det(L_1 - \lambda I) = \lambda^2 - 0.5\lambda - 0.6 = 0 \implies \lambda \approx 1.064, -0.564$. Dominant $|1.064| > 1 \implies$ **Unstable**.
- $L_2$: $\lambda^2 - 0.5\lambda - 0.5 = 0 \implies (\lambda - 1)(\lambda + 0.5) = 0$. Dominant $|1.0| = 1 \implies$ **Neutral** (converges to invariant state along eigenvector $(1, 1)^T$).
- $L_3$: $\lambda^2 - 0.5\lambda - 0.45 = 0 \implies \lambda \approx 0.966, -0.466$. Dominant $|0.966| < 1 \implies$ **Stable**.

</details>

## 7 · 3D Linear Difference Equation Asymptotic Analysis (Ex. 5.12 – Slide 38)

**Exercise 5.12:** Study the asymptotic behavior of the 3D model:
$$x_t = x_{t-1} - y_{t-1}$$
$$y_t = -x_{t-1} - 3\,y_{t-1} + z_{t-1}$$
$$z_t = y_{t-1} + z_{t-1}$$

**Tasks:**
1. Define matrix `A_512`.
2. Implement `eigenvalues_512()` returning the 3 analytical/numerical eigenvalues sorted ascending.
3. Identify the dominant eigenvector setting the long-term direction.

In [ ]:
A_512 = None  # TODO: 3x3 numpy array for Ex. 5.12


def eigenvalues_512():
    '''Returns the sorted eigenvalues of A_512.'''
    # TODO: implement
    raise NotImplementedError


def dominant_eigenvector_512():
    '''Returns normalized dominant eigenvector of A_512.'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests
assert A_512 is not None and A_512.shape == (3, 3)
evals = eigenvalues_512()
# Analytical roots: lambda = 1, and -1 +- sqrt(6)
exact_evals = sorted([1.0, -1.0 + np.sqrt(6.0), -1.0 - np.sqrt(6.0)])
assert np.allclose(evals, exact_evals), f"Expected {exact_evals}, got {evals}"

v_dom = dominant_eigenvector_512()
lam_dom = -1.0 - np.sqrt(6.0)
assert np.allclose(abs(A_512 @ v_dom), abs(lam_dom * v_dom))
print("✓ Ex. 5.12 3D asymptotic analysis")
print(f"Eigenvalues: {evals}")
print(f"Dominant eigenvalue: {lam_dom:.4f} (magnitude = {abs(lam_dom):.4f} > 1 -> Unstable)")

<details><summary>Hint: Characteristic Polynomial</summary>

$$\det(A - \lambda I) = (1 - \lambda)(\lambda^2 + 2\lambda - 5) = 0$$
Roots: $\lambda_1 = 1$, $\lambda_{2,3} = -1 \pm \sqrt{6} \approx 1.4495, -3.4495$.
The dominant eigenvalue is $\lambda_3 = -1 - \sqrt{6} \approx -3.4495$.
Because $|\lambda_3| > 1$ and is negative, trajectories oscillate and exponentially explode along $\mathbf{v}_3$.

</details>

## 8 · 1D Linear Stability Analysis (Exercise 5.15 – Slides 40–43, 45)

Linear stability analysis investigates small perturbations $\Delta x_t$ around an equilibrium $x_{eq}$:
$$\Delta x_t \approx F'(x_{eq})\,\Delta x_{t-1}$$
- $|F'(x_{eq})| < 1$: Stable
- $|F'(x_{eq})| > 1$: Unstable
- $|F'(x_{eq})| = 1$: Inconclusive (Lyapunov / borderline)

**Exercise 5.15:** Consider $(a > 0, b > 0)$:
$$x_t = x_{t-1} + a\,\sin(b\,x_{t-1})$$
Conduct linear stability analysis at $x_{eq} = 0$.

**Tasks:**
1. Implement `F_515(x, a, b)` and its derivative `dF_515(x, a, b)`.
2. Implement `is_stable_1d(dF_at_eq)` returning `True` if $|F'| < 1$, `False` if $|F'| > 1$, and `None` if borderline.

In [ ]:
def F_515(x, a, b):
    '''Iterative map x_t = x + a * sin(b * x).'''
    # TODO: implement
    raise NotImplementedError


def dF_515(x, a, b):
    '''Derivative F'(x) = dF/dx.'''
    # TODO: implement
    raise NotImplementedError


def is_stable_1d(dF_at_eq, atol=1e-6):
    '''Returns True (stable), False (unstable), or None (inconclusive).'''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests: Ex. 5.15
assert np.isclose(F_515(0.0, 1.0, 2.0), 0.0), "x_eq = 0 must be an equilibrium"
for a, b in [(0.5, 1.0), (1.0, 2.0), (0.1, 0.2)]:
    slope = dF_515(0.0, a, b)
    assert np.isclose(slope, 1.0 + a * b)
    assert is_stable_1d(slope) is False, f"Since a>0, b>0, 1+ab > 1 -> unstable, got {slope}"

assert is_stable_1d(0.5) is True
assert is_stable_1d(1.0) is None
print("✓ Ex. 5.15 linear stability analysis (x_eq = 0 is unstable)")

# Cobweb visual confirmation: perturbation diverges away from 0
f_test = lambda x: F_515(x, a=0.8, b=1.5)
plot_cobweb(f_test, cobweb_path(f_test, 0.05, 15), (-1.5, 1.5), title="Ex. 5.15: Cobweb spirals away from x=0");

<details><summary>Hint: Ex. 5.15</summary>

$$F'(x) = 1 + a b \cos(b x) \implies F'(0) = 1 + ab$$
Since $a > 0$ and $b > 0$, $ab > 0$, hence $F'(0) = 1 + ab > 1$. Therefore, the equilibrium point $x_{eq} = 0$ is always **unstable**.

</details>

## 9 · 2D Nonlinear Stability Analysis & Jacobian Matrix (Exercise 5.16 – Slide 45)

For multi-variable systems $\mathbf{x}_t = F(\mathbf{x}_{t-1})$, the Jacobian matrix at equilibrium $\mathbf{x}_{eq}$ is:
$$J = \left(\frac{\partial F_i}{\partial x_j}\right)_{\mathbf{x} = \mathbf{x}_{eq}}, \qquad \Delta \mathbf{x}_t \approx J\,\Delta \mathbf{x}_{t-1}$$

**Exercise 5.16:**
$$x_t = x_{t-1} + 2\,x_{t-1}(1 - x_{t-1}) - x_{t-1}\,y_{t-1}$$
$$y_t = y_{t-1} + 2\,y_{t-1}(1 - y_{t-1}) - x_{t-1}\,y_{t-1}$$

**Tasks:**
1. Implement `equilibria_516()` returning all 4 equilibria: $(0,0), (0,1), (1,0), (2/3, 2/3)$.
2. Implement `jacobian_516(x, y)`.
3. Implement `stability_516_positive_equilibrium()` returning the Jacobian, eigenvalues, and classification at $(2/3, 2/3)$.

In [ ]:
def equilibria_516():
    '''Returns list of all (x, y) equilibrium points of model 5.16.'''
    # TODO: implement
    raise NotImplementedError


def jacobian_516(x, y):
    '''Returns the 2x2 Jacobian matrix evaluated at (x, y).'''
    # TODO: implement
    raise NotImplementedError


def stability_516_positive_equilibrium():
    '''
    Returns (J, eigenvalues, classification) for the positive equilibrium (2/3, 2/3).
    Classification is one of: "stable point", "unstable point", "saddle point",
    "stable spiral focus", "unstable spiral focus", "neutral center", or "neutral point".
    '''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests: Ex. 5.16
eqs = sorted(equilibria_516())
expected_eqs = sorted([(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (2.0 / 3.0, 2.0 / 3.0)])
assert len(eqs) == 4, f"Expected 4 equilibria, got {len(eqs)}"
for eq, exp_eq in zip(eqs, expected_eqs):
    assert np.allclose(eq, exp_eq), f"Expected {exp_eq}, got {eq}"

J_pos, eigs_pos, cls_pos = stability_516_positive_equilibrium()
J_expected = np.array([[-1.0 / 3.0, -2.0 / 3.0], [-2.0 / 3.0, -1.0 / 3.0]])
assert np.allclose(J_pos, J_expected), f"Expected J = \n{J_expected},\ngot \n{J_pos}"
assert np.allclose(sorted(eigs_pos), [-1.0, 1.0 / 3.0]), f"Expected eigs [-1, 1/3], got {eigs_pos}"
assert cls_pos in ["neutral point", "neutral center"], f"Expected neutral classification, got {cls_pos}"
print("✓ Ex. 5.16 equilibria, Jacobian and stability")
print(f"Eigenvalues at (2/3, 2/3): {eigs_pos} -> dominant |lambda| = 1.0 ({cls_pos})")

# Phase space around (2/3, 2/3)
step_516 = lambda s: np.array([s[0] + 2*s[0]*(1 - s[0]) - s[0]*s[1], s[1] + 2*s[1]*(1 - s[1]) - s[0]*s[1]])
nearby_ics = [[0.6, 0.6], [0.7, 0.65], [0.65, 0.7], [0.55, 0.65]]
trajs_516 = [simulate(step_516, ic, 40) for ic in nearby_ics]
ax = plot_phase(trajs_516, title="Ex. 5.16: Trajectories Near (2/3, 2/3)")
ax.plot(2/3, 2/3, "r*", ms=12, label="Eq (2/3, 2/3)")
ax.legend();

<details><summary>Hint: Ex. 5.16 Derivation</summary>

1. Rewrite the equations:
   $$F_1(x, y) = 3x - 2x^2 - xy, \quad F_2(x, y) = 3y - 2y^2 - xy$$
2. Jacobian:
   $$J = \begin{pmatrix} 3 - 4x - y & -x \\ -y & 3 - 4y - x \end{pmatrix}$$
3. At $(2/3, 2/3)$:
   $$J = \begin{pmatrix} -1/3 & -2/3 \\ -2/3 & -1/3 \end{pmatrix}$$
   Characteristic equation: $(-1/3 - \lambda)^2 - (-2/3)^2 = 0 \implies (\lambda + 1/3)^2 = 4/9 \implies \lambda = -1, 1/3$.
4. Since dominant $ert{}\lambdaert{} = ert{}-1ert{} = 1.0$, this is a borderline Lyapunov stable / neutral point.

</details>

## 10 · 2D Stability Analysis: Predator–Prey Type Model (Exercise 5.17 – Slide 45)

**Exercise 5.17:** Consider
$$x_t = x_{t-1}\,y_{t-1}$$
$$y_t = y_{t-1}\,(x_{t-1} - 1)$$

**Tasks:**
1. Implement `jacobian_517(x, y)`.
2. Implement `analyze_equilibrium_517(eq)` returning `(J, eigenvalues, classification)`.
3. Discuss the stability of $(0, 0)$ and $(2, 1)$.

In [ ]:
def jacobian_517(x, y):
    '''Returns the 2x2 Jacobian matrix of model 5.17 at (x, y).'''
    # TODO: implement
    raise NotImplementedError


def analyze_equilibrium_517(eq):
    '''
    Returns (J, eigenvalues, classification) for a given equilibrium eq=(x, y).
    Classification is e.g. "neutral point" or "saddle point".
    '''
    # TODO: implement
    raise NotImplementedError

In [ ]:
# Tests: Ex. 5.17
J_00, eigs_00, cls_00 = analyze_equilibrium_517((0.0, 0.0))
assert np.allclose(J_00, [[0.0, 0.0], [0.0, -1.0]])
assert np.allclose(sorted(eigs_00), [-1.0, 0.0])
assert "neutral" in cls_00.lower() or "stable" in cls_00.lower()

J_21, eigs_21, cls_21 = analyze_equilibrium_517((2.0, 1.0))
assert np.allclose(J_21, [[1.0, 2.0], [1.0, 1.0]])
exact_eigs_21 = sorted([1.0 + np.sqrt(2.0), 1.0 - np.sqrt(2.0)])
assert np.allclose(sorted(eigs_21), exact_eigs_21)
assert "saddle" in cls_21.lower() or "unstable" in cls_21.lower()
print("✓ Ex. 5.17 Jacobian & Stability Analysis")
print(f"At (0, 0): lambda = {eigs_00} -> {cls_00}")
print(f"At (2, 1): lambda = {eigs_21} -> {cls_21}")

# Phase portrait near the saddle point (2, 1)
step_517 = lambda s: np.array([s[0] * s[1], s[1] * (s[0] - 1.0)])
trajs_517 = [simulate(step_517, ic, 8) for ic in [[2.01, 1.0], [1.99, 1.0], [2.0, 1.01], [2.0, 0.99]]]
ax = plot_phase(trajs_517, title="Ex. 5.17: Saddle Point Dynamics near (2, 1)")
ax.plot(2, 1, "r*", ms=12, label="Saddle (2, 1)")
ax.set_xlim(1.5, 2.5); ax.set_ylim(0.5, 1.5); ax.legend();

<details><summary>Hint: Ex. 5.17 Derivation</summary>

$$J = \begin{pmatrix} y & x \\ y & x - 1 \end{pmatrix}$$
- At $(0, 0)$: $J = \begin{pmatrix} 0 & 0 \\ 0 & -1 \end{pmatrix}$, eigenvalues $\lambda = 0, -1$. Dominant $ert{}\lambda_dert{} = 1 \implies$ Lyapunov/neutral.
- At $(2, 1)$: $J = \begin{pmatrix} 1 & 2 \\ 1 & 1 \end{pmatrix}$, $\det(J - \lambda I) = (1 - \lambda)^2 - 2 = 0 \implies \lambda = 1 \pm \sqrt{2} \approx 2.414, -0.414$.
  Since $ert{}2.414ert{} > 1$ (expanding) and $ert{}-0.414ert{} < 1$ (contracting), $(2, 1)$ is a **saddle point**.

</details>

## Notes on the Slides & Big Picture Summary

| Concept | Scalar Form ($1$D) | Multi-Dimensional ($n$D) |
|---|---|---|
| **Linearization** | $F'(x_{eq})$ | Jacobian matrix $J = \left(\frac{\partial F_i}{\partial x_j}\right)_{x_{eq}}$ |
| **Local Dynamics** | $\Delta x_t \approx F'(x_{eq})\,\Delta x_{t-1}$ | $\Delta \mathbf{x}_t \approx J\,\Delta \mathbf{x}_{t-1}$ |
| **Stability Test** | $ert{}F'(x_{eq})ert{} < 1$ (Stable)<br>$ert{}F'(x_{eq})ert{} > 1$ (Unstable) | $ert{}\lambda_d(J)ert{} < 1$ (Stable)<br>$ert{}\lambda_d(J)ert{} > 1$ (Unstable) |

- **Simultaneous vs. Sequential updates:** In discrete-time systems, all state equations at time $t$ evaluate the states from time $t-1$. Updating sequentially inside a loop without temporary variables introduces an artificial damping bug (spiral inward instead of closed orbit).
- **Continuous vs. Discrete stability:** In discrete time, stability requires $ert{}\lambdaert{} < 1$ (inside the unit circle in the complex plane), NOT $\text{Re}(\lambda) < 0$ (which is for continuous-time ODEs).
- **Borderline $ert{}\lambda_dert{} = 1$:** Linear stability is inconclusive; higher-order nonlinear terms dictate the true asymptotic behavior.